# 02 — Phase 2 (T_t transfer control) + Phase 2b (activation Q metrics)

Phase 2 is cheap and MUST be reported to the channel before Phase 3 (notebook 03) starts — it decides how `ΔR_t` can be read (plan §2 reading rule). Phase 2b gates nothing downstream.

In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os, json
from pathlib import Path

# Private repo: reads a token from Colab's own Secrets store (key icon,
# left sidebar) — add one named GITHUB_TOKEN (a GitHub PAT with repo read
# access) before running this cell. The token is never written to this
# notebook's source and this cell never prints it.
from google.colab import userdata
try:
    _token = userdata.get('GITHUB_TOKEN')
except Exception:
    _token = None
if not _token:
    raise RuntimeError(
        'Add a GITHUB_TOKEN secret (key icon, left sidebar) with repo read '
        'access to WYR186/RLVR, enable notebook access for it, then re-run.')
REPO_URL = f'https://{_token}@github.com/WYR186/RLVR.git'
REPO_DIR = '/content/RLVR'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
# strip the token back out of the stored remote URL immediately — no need
# to leave it sitting in .git/config for the rest of the session
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin',
                'https://github.com/WYR186/RLVR.git'], check=True)
del _token, REPO_URL  # don't leave the token bound in the notebook's live namespace

EXP2_DIR = f'{REPO_DIR}/experiment 2'
sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path — pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

CONFIG = json.load(open(f'{EXP2_DIR}/exp2_colab_config.json'))
DATA_DIR = Path(EXP2_DIR) / 'data'
MODEL_ID, MODEL_REVISION = CONFIG['model_id'], CONFIG['model_revision']
DATASET_REVISION = CONFIG['dataset']['revision']
print('config loaded:', CONFIG['experiment'])
print('merge note:', CONFIG['merge_note'])

In [ ]:
splits = json.load(open(DATA_DIR / 'exp2_colab_splits.json'))
eval_rows = guru_data.dataset_rows_for(
    'b', 'eval', splits, MODEL_ID, MODEL_REVISION, DATASET_REVISION)
print('stage-B eval set:', len(eval_rows))

RUN_DIR = f'{EXP2_DIR}/../eaaj-pilot/outputs/exp2_colab_guru_math7b_group8_REPLACE_WITH_HASH'
STAGE_A_DIR = f'{RUN_DIR}/stage_a'
sa = CONFIG['stage_a']

## Phase 2 — T_t (zero-shot, no stage-2 training)

In [ ]:
transfer = pipeline.run_transfer_T(
    MODEL_ID, CONFIG['peft'], STAGE_A_DIR, sa['checkpoint_steps'],
    eval_rows, f'{RUN_DIR}/analysis/transfer_T.json', revision=MODEL_REVISION)
print(transfer)
print('Report T_t to the channel before starting notebook 03.')

## Phase 2b — activation Q metrics (frozen probe, cross-domain top-up if needed)

In [ ]:
m = CONFIG['measurement']
probe_rows = guru_data.dataset_rows_for(
    'probe', None, splits, MODEL_ID, MODEL_REVISION, DATASET_REVISION)
probe_prompts = [r['prompt'] for r in probe_rows]
print('probe size:', len(probe_prompts), '(requested', m['probe_questions'], ')',
      '| stage-B rows:', len(splits['probe_stage_b_ids']),
      '| stage-A topup:', len(splits['probe_stage_a_topup_ids']))
if len(probe_prompts) < 3584:
    print('WARNING: probe smaller than hidden dim (3584) — erank magnitudes are '
          'sample-truncated; report n_probe alongside every value (plan §1).')

(Path(RUN_DIR) / 'measurements').mkdir(parents=True, exist_ok=True)
q_by_ckpt = {}
for step in sa['checkpoint_steps']:
    ckpt_dir = f'{STAGE_A_DIR}/ckpt-{step}'
    q = pipeline.measure_checkpoint_q(
        MODEL_ID, CONFIG['peft'], ckpt_dir, probe_prompts,
        layers=tuple(m['layers']), revision=MODEL_REVISION, batch_size=m['batch_size'])
    q_by_ckpt[step] = q
    json.dump(q, open(f'{RUN_DIR}/measurements/metrics_ckpt{step}.json', 'w'), indent=1, default=str)
    print('measured ckpt', step)

## ckpt-0 within-run identity re-check (plan §1 — not a cross-model comparison)

In [ ]:
q_ckpt0_again = pipeline.measure_checkpoint_q(
    MODEL_ID, CONFIG['peft'], f'{STAGE_A_DIR}/ckpt-0', probe_prompts,
    layers=tuple(m['layers']), revision=MODEL_REVISION, batch_size=m['batch_size'])
l = m['layers'][0]
erank_1 = q_by_ckpt[0]['per_layer'][f'layer{l}']['erank']
erank_2 = q_ckpt0_again['per_layer'][f'layer{l}']['erank']
delta = abs(erank_1 - erank_2)
print(f'ckpt-0 identity re-check, layer {l}: erank {erank_1:.6f} vs {erank_2:.6f}, delta={delta:.2e}')
if delta > 1e-4:
    print('WARNING: measurement contract may have drifted between the two ckpt-0 passes — investigate before trusting Phase 2b.')

## Commit reminder

Commit `analysis/transfer_T.json` and `measurements/metrics_ckpt*.json`, prefix `exp2-colab:`.